# Fruit Ninja: control por nariz o dedo

**N**: nariz · **D**: punta del dedo índice · **Q**: salir · **R**: reiniciar después del fin de partida.

Las frutas y la bandana se leen desde GitHub al iniciar el juego; hace falta conexión a Internet. Los modelos de MediaPipe se guardan localmente tras descargarse una vez. Si bomba.png no está publicada, se muestra una bomba dibujada.

### Dependencias (solo la primera vez)

En una celda de Jupyter ejecuta `%pip install mediapipe opencv-python numpy`. No hace falta reinstalarlas cada vez que juegues.

In [25]:
from pathlib import Path
from urllib.request import urlretrieve

# Los modelos de MediaPipe se guardan localmente; el cuaderno los reutiliza tras la primera descarga.
MODEL_FACE = Path("face_landmarker.task")
MODEL_HAND = Path("hand_landmarker.task")
MODEL_URLS = {
    MODEL_FACE: "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    MODEL_HAND: "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
}

# Las imagenes se leen directamente desde la carpeta publica del repositorio.
REPO_IMAGES = "https://raw.githubusercontent.com/bezz75/VC-practica/main/P5/FruitNinja/images"
FRUIT_URLS = {
    "paraguayo.png": f"{REPO_IMAGES}/paraguayo.png",
    "manzana.png": f"{REPO_IMAGES}/manzana.png",
    "naranja.png": f"{REPO_IMAGES}/naranja.png",
    "sandia.png": f"{REPO_IMAGES}/sandia.png",
}
BOMB_URL = f"{REPO_IMAGES}/bomba.png"
HEADBAND_URL = f"{REPO_IMAGES}/cinta.png"

# Solo los modelos .task se descargan a disco, porque MediaPipe necesita una ruta local.
for archivo, url in MODEL_URLS.items():
    if not archivo.exists():
        print(f"Descargando {archivo.name}...")
        # Se descarga a un archivo temporal para no dejar un modelo corrupto si se corta.
        temporal = archivo.with_suffix(".tmp")
        urlretrieve(url, temporal)
        temporal.replace(archivo)

print("Modelos listos. Los PNG se cargaran desde GitHub al iniciar el juego.")

Modelos listos. Los PNG se cargaran desde GitHub al iniciar el juego.


In [26]:
# Utilidades para guardar objetos, suavizar el control y procesar imagenes.
from collections import deque
from dataclasses import dataclass
from urllib.error import HTTPError
from urllib.request import urlopen
import math
import random
import time

import cv2
import mediapipe as mp
import numpy as np

# Ajustes del juego.
ANCHO_CAMARA = 640
ALTO_CAMARA = 480
DURACION = 60.0
VIDAS_INICIALES = 3
INTERVALO_FRUTAS = 0.9
INTERVALO_BOMBAS = 4.5
GRAVEDAD = 1050.0
ALPHA = 0.35
LARGO_ESTELA = 14
MARGEN_CORTE = 8
TAMANO_MAXIMO_SPRITE = 160
VELOCIDAD_MINIMA_CORTE = 6  # pixeles por fotograma; evita cortar con el cursor quieto

# Color de las particulas al cortar cada fruta (BGR).
FRUIT_COLORS = {
    "paraguayo.png": (40, 40, 220),
    "manzana.png": (30, 30, 220),
    "naranja.png": (0, 140, 255),
    "sandia.png": (40, 180, 50),
}

# Decodifica PNG desde GitHub en memoria; solo bomba.png puede faltar sin detener el juego.
def cargar_png_desde_github(url, opcional=False):
    try:
        with urlopen(url, timeout=30) as respuesta:
            datos_png = np.frombuffer(respuesta.read(), dtype=np.uint8)
    except HTTPError as error:
        if opcional and error.code == 404:
            print("No se encontró bomba.png en GitHub; se usará la bomba dibujada.")
            return None
        raise RuntimeError(f"GitHub respondió HTTP {error.code} al acceder a {url}") from error
    except Exception as error:
        raise RuntimeError(f"No se pudo leer el PNG desde GitHub: {url}: {error}") from error

    sprite = cv2.imdecode(datos_png, cv2.IMREAD_UNCHANGED)
    if sprite is None:
        raise RuntimeError(f"La URL no devolvió un PNG válido: {url}")
    return sprite

# Reduce los sprites grandes manteniendo su proporción y transparencia.
def reducir_sprite(sprite, tamano_maximo):
    alto, ancho = sprite.shape[:2]
    escala = min(1.0, tamano_maximo / max(alto, ancho))
    if escala < 1.0:
        nuevo_tamano = (max(1, int(ancho * escala)), max(1, int(alto * escala)))
        sprite = cv2.resize(sprite, nuevo_tamano, interpolation=cv2.INTER_AREA)
    return sprite

# Lee las frutas directamente de sus URL y deja las versiones pequeñas en memoria.
FRUIT_SPRITES = [
    (
        reducir_sprite(cargar_png_desde_github(url), TAMANO_MAXIMO_SPRITE),
        FRUIT_COLORS.get(nombre, (0, 200, 220)),
    )
    for nombre, url in FRUIT_URLS.items()
]

# Si el PNG de la bomba existe, también se carga y se prepara para el juego.
BOMB_SPRITE = cargar_png_desde_github(BOMB_URL, opcional=True)
if BOMB_SPRITE is not None:
    BOMB_SPRITE = reducir_sprite(BOMB_SPRITE, TAMANO_MAXIMO_SPRITE)

# Recorta el espacio transparente de la cinta para ajustar su forma a la frente.
HEADBAND_SPRITE = cargar_png_desde_github(HEADBAND_URL)
if HEADBAND_SPRITE.ndim != 3 or HEADBAND_SPRITE.shape[2] != 4:
    raise RuntimeError("La imagen de la cinta debe ser un PNG con transparencia.")

mascara_cinta = (HEADBAND_SPRITE[:, :, 3] > 8).astype("uint8")
coords_cinta = cv2.findNonZero(mascara_cinta)
if coords_cinta is None:
    raise RuntimeError("La imagen de la cinta está vacía.")
x_cinta, y_cinta, w_cinta, h_cinta = cv2.boundingRect(coords_cinta)
HEADBAND_SPRITE = HEADBAND_SPRITE[
    y_cinta:y_cinta + h_cinta, x_cinta:x_cinta + w_cinta
]
if HEADBAND_SPRITE.shape[1] > 1200:
    escala_cinta = 1200 / HEADBAND_SPRITE.shape[1]
    HEADBAND_SPRITE = cv2.resize(
        HEADBAND_SPRITE,
        (1200, max(1, int(HEADBAND_SPRITE.shape[0] * escala_cinta))),
        interpolation=cv2.INTER_AREA,
    )

# Estructuras para guardar el estado de frutas, bombas y particulas.
@dataclass
class Fruta:
    x: float
    y: float
    vx: float
    vy: float
    radio: int
    sprite: object
    color: tuple

@dataclass
class Bomba:
    x: float
    y: float
    vx: float
    vy: float
    radio: int

@dataclass
class Particula:
    x: float
    y: float
    vx: float
    vy: float
    vida: float
    vida_max: float
    color: tuple
    radio: int

# Crea una fruta abajo de la pantalla con velocidad inicial hacia arriba.
def crear_fruta(ancho, alto):
    sprite, color = random.choice(FRUIT_SPRITES)
    radio = random.randint(30, 40)
    return Fruta(
        random.randint(radio, ancho - radio),
        alto + radio,
        random.uniform(-120, 120),
        -random.uniform(820, 950),
        radio,
        sprite,
        color,
    )

# Genera bombas con movimiento parecido al de las frutas.
def crear_bomba(ancho, alto):
    radio = random.randint(25, 33)
    return Bomba(
        random.randint(radio, ancho - radio),
        alto + radio,
        random.uniform(-100, 100),
        -random.uniform(840, 960),
        radio,
    )

# Aplica velocidad y gravedad, y elimina objetos fuera de la zona visible.
def actualizar_objetos(objetos, dt, ancho, alto):
    for objeto in objetos:
        objeto.x += objeto.vx * dt
        objeto.y += objeto.vy * dt
        objeto.vy += GRAVEDAD * dt
    objetos[:] = [
        objeto for objeto in objetos
        if objeto.y - objeto.radio <= alto + 60
        and -objeto.radio <= objeto.x <= ancho + objeto.radio
    ]

# Guarda cada sprite ya redimensionado para no repetir cv2.resize en todos los fotogramas.
CACHE_SPRITES = {}

def sprite_escalado(sprite, diametro):
    clave = (id(sprite), diametro)
    if clave not in CACHE_SPRITES:
        alto_img, ancho_img = sprite.shape[:2]
        factor = diametro / max(alto_img, ancho_img)
        nuevo_tamano = (max(1, int(ancho_img * factor)), max(1, int(alto_img * factor)))
        interpolacion = cv2.INTER_AREA if factor < 1 else cv2.INTER_LINEAR
        CACHE_SPRITES[clave] = cv2.resize(sprite, nuevo_tamano, interpolation=interpolacion)
    return CACHE_SPRITES[clave]

# Superpone un PNG sobre la webcam y mezcla los pixeles usando su transparencia.
def dibujar_png(frame, sprite, centro_x, centro_y, diametro):
    sprite = sprite_escalado(sprite, diametro)
    nuevo_alto, nuevo_ancho = sprite.shape[:2]

    x0 = int(centro_x - nuevo_ancho / 2)
    y0 = int(centro_y - nuevo_alto / 2)
    x1, y1 = max(0, x0), max(0, y0)
    x2 = min(frame.shape[1], x0 + nuevo_ancho)
    y2 = min(frame.shape[0], y0 + nuevo_alto)
    if x1 >= x2 or y1 >= y2:
        return

    recorte = sprite[y1 - y0:y2 - y0, x1 - x0:x2 - x0]
    zona = frame[y1:y2, x1:x2]

    if recorte.ndim == 3 and recorte.shape[2] == 4:
        alpha = recorte[:, :, 3:4].astype("float32") / 255.0
        frente = recorte[:, :, :3].astype("float32")
        fondo = zona.astype("float32")
        zona[:] = (alpha * frente + (1 - alpha) * fondo).astype("uint8")
    elif recorte.ndim == 2:
        zona[:] = cv2.cvtColor(recorte, cv2.COLOR_GRAY2BGR)
    else:
        zona[:] = recorte[:, :, :3]

# Ajusta tamaño, posicion e inclinacion de la bandana a los puntos de la cara.
def dibujar_cinta(frame, landmarks):
    """Coloca la cinta sobre la frente y la gira según la inclinación de la cabeza."""
    if len(landmarks) <= 356:
        return

    alto, ancho = frame.shape[:2]
    a = (landmarks[127].x * ancho, landmarks[127].y * alto)
    b = (landmarks[356].x * ancho, landmarks[356].y * alto)
    if a[0] > b[0]:
        a, b = b, a

    ancho_cara = math.hypot(b[0] - a[0], b[1] - a[1])
    if ancho_cara < 12:
        return

    # El punto 10 marca la parte alta de la frente; 105 y 334 están junto a las cejas.
    frente_arriba = landmarks[10]
    ceja_izq = landmarks[105]
    ceja_der = landmarks[334]
    centro_x = (a[0] + b[0]) / 2
    centro_y = alto * (
        frente_arriba.y + (ceja_izq.y + ceja_der.y) / 2
    ) / 2

    sprite_h, sprite_w = HEADBAND_SPRITE.shape[:2]
    objetivo_w = int(min(ancho_cara * 1.35, ancho * 0.72, 420))
    objetivo_w = max(24, objetivo_w)
    factor = objetivo_w / sprite_w
    nuevo_w = max(1, int(sprite_w * factor))
    nuevo_h = max(1, int(sprite_h * factor))
    cinta = cv2.resize(
        HEADBAND_SPRITE, (nuevo_w, nuevo_h),
        interpolation=cv2.INTER_AREA if factor < 1 else cv2.INTER_LINEAR,
    )

    inclinacion = math.degrees(math.atan2(b[1] - a[1], b[0] - a[0]))
    matriz = cv2.getRotationMatrix2D((nuevo_w / 2, nuevo_h / 2), -inclinacion, 1.0)
    rot_w = int(nuevo_h * abs(matriz[0, 1]) + nuevo_w * abs(matriz[0, 0]))
    rot_h = int(nuevo_h * abs(matriz[0, 0]) + nuevo_w * abs(matriz[0, 1]))
    matriz[0, 2] += rot_w / 2 - nuevo_w / 2
    matriz[1, 2] += rot_h / 2 - nuevo_h / 2
    cinta = cv2.warpAffine(
        cinta, matriz, (rot_w, rot_h), flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0),
    )

    # El PNG tiene el lazo a la derecha; desplazamos el conjunto para centrar el paño
    # sobre la frente en vez de centrar el ancho total del dibujo.
    desplazamiento = objetivo_w * 0.11
    angulo_rad = math.radians(inclinacion)
    centro_cinta_x = centro_x + desplazamiento * math.cos(angulo_rad)
    centro_cinta_y = centro_y + desplazamiento * math.sin(angulo_rad)
    x0 = int(centro_cinta_x - rot_w / 2)
    y0 = int(centro_cinta_y - rot_h / 2)
    x1, y1 = max(0, x0), max(0, y0)
    x2, y2 = min(ancho, x0 + rot_w), min(alto, y0 + rot_h)
    if x1 >= x2 or y1 >= y2:
        return

    recorte = cinta[y1 - y0:y2 - y0, x1 - x0:x2 - x0]
    zona = frame[y1:y2, x1:x2]
    alpha = recorte[:, :, 3:4].astype("float32") / 255.0
    zona[:] = (
        alpha * recorte[:, :, :3].astype("float32")
        + (1 - alpha) * zona.astype("float32")
    ).astype("uint8")


# Dibuja los sprites de frutas y las bombas sobre cada fotograma.
def dibujar_frutas(frame, frutas):
    for fruta in frutas:
        dibujar_png(
            frame, fruta.sprite, int(fruta.x), int(fruta.y), fruta.radio * 2
        )

def dibujar_bombas(frame, bombas):
    for bomba in bombas:
        dibujar_png(
            frame, BOMB_SPRITE, int(bomba.x), int(bomba.y), bomba.radio * 2
        )
    

# Calcula si el movimiento del cursor ha pasado cerca de una fruta o bomba.
def distancia_punto_segmento(punto, inicio, fin):
    px, py = punto
    ax, ay = inicio
    bx, by = fin
    vx, vy = bx - ax, by - ay
    longitud2 = vx * vx + vy * vy
    if longitud2 == 0:
        qx, qy = ax, ay
    else:
        t = ((px - ax) * vx + (py - ay) * vy) / longitud2
        t = max(0.0, min(1.0, t))
        qx, qy = ax + t * vx, ay + t * vy
    return math.hypot(px - qx, py - qy)

# Crea particulas para dar feedback visual al cortar una fruta o tocar una bomba.
def crear_explosion(x, y, color, cantidad=18):
    particulas = []
    for _ in range(cantidad):
        angulo = random.uniform(0, 2 * math.pi)
        velocidad = random.uniform(90, 280)
        vida = random.uniform(0.25, 0.55)
        particulas.append(Particula(
            x, y,
            math.cos(angulo) * velocidad,
            math.sin(angulo) * velocidad,
            vida, vida, color, random.randint(3, 7),
        ))
    return particulas

def actualizar_particulas(particulas, dt):
    for p in particulas:
        p.x += p.vx * dt
        p.y += p.vy * dt
        p.vy += GRAVEDAD * 0.45 * dt
        p.vx *= 0.985
        p.vida -= dt
    particulas[:] = [p for p in particulas if p.vida > 0]

def dibujar_particulas(frame, particulas):
    for p in particulas:
        intensidad = max(0.0, p.vida / p.vida_max)
        color = tuple(int(c * intensidad) for c in p.color)
        cv2.circle(
            frame, (int(p.x), int(p.y)), max(1, int(p.radio * intensidad)),
            color, -1, cv2.LINE_AA,
        )

In [27]:
if not MODEL_FACE.exists() or not MODEL_HAND.exists():
    raise FileNotFoundError("Ejecuta primero la celda que descarga los modelos.")

# Abre la webcam y solicita la resolucion de trabajo del juego.
camera = cv2.VideoCapture(0)
if not camera.isOpened():
    camera.release()
    raise RuntimeError("No se pudo abrir la webcam.")

camera.set(cv2.CAP_PROP_FRAME_WIDTH, ANCHO_CAMARA)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, ALTO_CAMARA)

# Configura los detectores en modo VIDEO para procesar fotogramas sucesivos.
face_options = mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_FACE)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_faces=1,
)
hand_options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_HAND)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_hands=1,
)

NOMBRE_VENTANA = "Fruit Ninja - nariz o dedo"

face_tracker = None
hand_tracker = None

try:
    # Inicializa los detectores y prepara el estado inicial de la partida.
    face_tracker = mp.tasks.vision.FaceLandmarker.create_from_options(face_options)
    hand_tracker = mp.tasks.vision.HandLandmarker.create_from_options(hand_options)

    modo = None
    game_over = False
    puntos = 0
    vidas = VIDAS_INICIALES
    frutas = []
    bombas = []
    particulas = []
    estela = deque(maxlen=LARGO_ESTELA)
    punto_filtrado = None

    inicio_partida = None
    tiempo_anterior = time.perf_counter()
    timer_frutas = 0.0
    timer_bombas = 0.0
    # Reloj unico para MediaPipe: los timestamps deben crecer con el tiempo real
    # y no reiniciarse al empezar otra partida.
    reloj_inicio = time.monotonic()
    timestamp_ms = 0
    restante = DURACION
    mensaje = ""
    mensaje_hasta = 0.0

    # En cada vuelta captura, detecta el control, actualiza el juego y dibuja la pantalla.
    while True:
        ok, frame = camera.read()
        if not ok:
            print("No se pudo leer la webcam.")
            break

        # dt es el tiempo entre fotogramas y se limita para estabilizar la fisica.
        ahora_perf = time.perf_counter()
        dt = min(max(ahora_perf - tiempo_anterior, 0.0), 0.05)
        tiempo_anterior = ahora_perf

        frame = cv2.flip(frame, 1)
        alto, ancho = frame.shape[:2]
        raw_point = None
        face_landmarks = None
        timestamp_ms = max(timestamp_ms + 1, int((time.monotonic() - reloj_inicio) * 1000))

        # Se detecta la cara en ambos modos: controla la nariz o coloca la bandana.
        if modo is not None:
            imagen = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB),
            )
            resultado_cara = face_tracker.detect_for_video(imagen, timestamp_ms)
            if resultado_cara.face_landmarks:
                face_landmarks = resultado_cara.face_landmarks[0]

            if modo == "nariz" and not game_over and face_landmarks is not None:
                punta = face_landmarks[1]
                raw_point = (punta.x * ancho, punta.y * alto)
            elif modo == "dedo" and not game_over:
                resultado_mano = hand_tracker.detect_for_video(imagen, timestamp_ms)
                if resultado_mano.hand_landmarks:
                    punta = resultado_mano.hand_landmarks[0][8]
                    raw_point = (punta.x * ancho, punta.y * alto)

        # Actualiza objetos, trazo y puntuacion mientras la partida sigue activa.
        if modo is not None and not game_over:
            if raw_point is None:
                punto_filtrado = None
                estela.clear()
            elif punto_filtrado is None:
                punto_filtrado = raw_point
                estela.append((round(raw_point[0]), round(raw_point[1])))
            else:
                punto_filtrado = (
                    ALPHA * raw_point[0] + (1 - ALPHA) * punto_filtrado[0],
                    ALPHA * raw_point[1] + (1 - ALPHA) * punto_filtrado[1],
                )
                estela.append((round(punto_filtrado[0]), round(punto_filtrado[1])))

            if inicio_partida is not None:
                restante = max(0.0, DURACION - (time.monotonic() - inicio_partida))
            else:
                restante = DURACION

            timer_frutas += dt
            while timer_frutas >= INTERVALO_FRUTAS:
                frutas.append(crear_fruta(ancho, alto))
                timer_frutas -= INTERVALO_FRUTAS

            timer_bombas += dt
            while timer_bombas >= INTERVALO_BOMBAS:
                bombas.append(crear_bomba(ancho, alto))
                timer_bombas -= INTERVALO_BOMBAS

            actualizar_objetos(frutas, dt, ancho, alto)
            actualizar_objetos(bombas, dt, ancho, alto)

            # Solo se corta si el cursor se mueve; con el cursor quieto no hay corte.
            if (len(estela) >= 2 and math.hypot(estela[-1][0] - estela[-2][0],
                                                estela[-1][1] - estela[-2][1])
                    >= VELOCIDAD_MINIMA_CORTE):
                a, b = estela[-2], estela[-1]
                restantes = []
                for fruta in frutas:
                    if distancia_punto_segmento((fruta.x, fruta.y), a, b) <= fruta.radio + MARGEN_CORTE:
                        puntos += 10
                        particulas.extend(crear_explosion(fruta.x, fruta.y, fruta.color, 20))
                        mensaje, mensaje_hasta = "+10", time.monotonic() + 0.4
                    else:
                        restantes.append(fruta)
                frutas = restantes

                restantes = []
                for bomba in bombas:
                    if distancia_punto_segmento((bomba.x, bomba.y), a, b) <= bomba.radio + MARGEN_CORTE:
                        vidas -= 1
                        particulas.extend(crear_explosion(bomba.x, bomba.y, (30, 70, 255), 26))
                        mensaje, mensaje_hasta = "BOMBA: -1 vida", time.monotonic() + 0.6
                    else:
                        restantes.append(bomba)
                bombas = restantes

            if vidas <= 0 or restante <= 0:
                game_over = True
                frutas.clear()
                bombas.clear()

        elif modo is None:
            # En el menu el tiempo se muestra completo; en fin de partida se queda congelado.
            restante = DURACION

        # La bandana se dibuja primero para que frutas, bombas y trazo queden por delante.
        if face_landmarks is not None:
            dibujar_cinta(frame, face_landmarks)

        # Actualiza y dibuja frutas, bombas, particulas y trazo.
        actualizar_particulas(particulas, dt)
        dibujar_frutas(frame, frutas)
        dibujar_bombas(frame, bombas)
        dibujar_particulas(frame, particulas)

        for i in range(1, len(estela)):
            grosor = max(2, int(2 + 7 * i / len(estela)))
            cv2.line(frame, estela[i - 1], estela[i],
                     (0, 220, 120), grosor, cv2.LINE_AA)
        if estela:
            cv2.circle(frame, estela[-1], 12, (0, 255, 255), -1)
            cv2.circle(frame, estela[-1], 17, (0, 0, 0), 2)

        # Dibuja la pantalla de inicio o la informacion de la partida.
        if modo is None:
            capa = frame.copy()
            cv2.rectangle(capa, (20, 120), (ancho - 20, alto - 100), (15, 15, 15), -1)
            frame = cv2.addWeighted(capa, 0.72, frame, 0.28, 0)
            cv2.putText(frame, "FRUIT NINJA", (ancho // 2 - 150, alto // 2 - 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3, cv2.LINE_AA)
            cv2.putText(frame, "Pulsa N para usar la nariz",
                        (ancho // 2 - 180, alto // 2 + 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, "Pulsa D para usar el dedo indice",
                        (ancho // 2 - 200, alto // 2 + 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, "Q para salir",
                        (ancho // 2 - 70, alto // 2 + 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1)

        else:
            control = "Nariz (N/D cambia)" if modo == "nariz" else "Dedo indice (N/D cambia)"
            cv2.putText(frame, f"Puntos: {puntos}  Vidas: {vidas}  Tiempo: {int(restante)}s",
                        (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2)
            cv2.putText(frame, f"Control: {control}",
                        (15, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

            if modo == "nariz" and raw_point is None and not game_over:
                cv2.putText(frame, "No detecto la cara",
                            (15, 88), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
            if modo == "dedo" and raw_point is None and not game_over:
                cv2.putText(frame, "Muestra la mano a la camara",
                            (15, 88), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
            if time.monotonic() < mensaje_hasta:
                cv2.putText(frame, mensaje, (ancho // 2 - 80, 85),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 3)

            if game_over:
                capa = frame.copy()
                cv2.rectangle(capa, (20, 150), (ancho - 20, alto - 130), (15, 15, 15), -1)
                frame = cv2.addWeighted(capa, 0.72, frame, 0.28, 0)
                cv2.putText(frame, "FIN DE PARTIDA",
                            (ancho // 2 - 170, alto // 2 - 20),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)
                cv2.putText(frame, f"Puntuacion: {puntos} | R reinicia | Q sale",
                            (ancho // 2 - 210, alto // 2 + 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
            else:
                cv2.putText(frame, "N: nariz | D: dedo | Q: salir",
                            (15, alto - 18), cv2.FONT_HERSHEY_SIMPLEX,
                            0.55, (255, 255, 255), 2)

        # Muestra el fotograma y lee la tecla pulsada por el jugador.
        cv2.imshow(NOMBRE_VENTANA, frame)
        tecla = cv2.waitKey(1) & 0xFF

        # Q o Escape sale; tambien se sale al cerrar la ventana con la X.
        if tecla in (ord("q"), ord("Q"), 27):
            break
        if cv2.getWindowProperty(NOMBRE_VENTANA, cv2.WND_PROP_VISIBLE) < 1:
            break

        # N/D elige control y R reinicia al acabar.

        if not game_over and tecla in (ord("n"), ord("N"), ord("d"), ord("D")):
            nuevo_modo = "nariz" if tecla in (ord("n"), ord("N")) else "dedo"
            if modo != nuevo_modo:
                modo = nuevo_modo
                punto_filtrado = None
                estela.clear()
                if inicio_partida is None:
                    inicio_partida = time.monotonic()

        if game_over and tecla in (ord("r"), ord("R")):
            modo = None
            game_over = False
            puntos = 0
            vidas = VIDAS_INICIALES
            frutas.clear()
            bombas.clear()
            particulas.clear()
            estela.clear()
            punto_filtrado = None
            inicio_partida = None
            timer_frutas = 0.0
            timer_bombas = 0.0
            tiempo_anterior = time.perf_counter()

finally:
    # Libera la webcam, los detectores y las ventanas incluso al salir con error.
    camera.release()
    if face_tracker is not None:
        face_tracker.close()
    if hand_tracker is not None:
        hand_tracker.close()
    cv2.destroyAllWindows()
    cv2.waitKey(1)  # necesario en algunos sistemas (macOS) para que la ventana se cierre